[目录](./table_of_contents.ipynb)

# 自适应滤波

In [ ]:
%matplotlib inline

In [ ]:
#格式化书本
import book_format
book_format.set_style()

## 介绍

到目前为止，我们考虑了跟踪与我们的过程模型相关的行为良好的对象的问题。例如，我们可以使用恒定速度滤波器来跟踪直线运动的物体。只要物体以相对恒定的速度直线运动或者在运动轨迹和/或速度上变化非常缓慢，这个滤波器就会表现得非常好。现在我们考虑另一种情况，即跟踪一个机动目标，例如沿着公路行驶的汽车、飞行中的飞机等等。在这些情况下，滤波器表现得相当糟糕。或者，考虑一种情况，例如跟踪海洋中的帆船。即使我们对控制输入进行建模，我们也无法对风力或洋流进行建模。

解决这个问题的一种一阶方法是增加过程噪声 $\mathbf{Q}$，以考虑系统动力学的不可预测性。虽然这可以在提供非发散滤波器方面起作用，但结果通常远非最佳。较大的 $\mathbf{Q}$ 会导致滤波器更加强调测量中的噪声。我们很快会看到一个例子。

在本章中，我们将讨论 *自适应滤波器* 的概念。当它检测到过程模型无法解释的动态时，该滤波器将自行 *适应*。我将从一个问题的例子开始，然后讨论和实现各种自适应滤波器。

## 操纵目标

我们需要一个操纵目标的模拟。我将实现一个简单的带有转向输入的 2D 模型。您可以提供一个新的速度和/或方向，它将修改其状态以匹配。

In [ ]:
from math import sin, cos, radians

def angle_between(x, y):
  return min(y-x, y-x+360, y-x-360, key=abs)

In [ ]:
class ManeuveringTarget(object): 
    def __init__(self, x0, y0, v0, heading):
        self.x = x0
        self.y = y0
        self.vel = v0
        self.hdg = heading
        
        self.cmd_vel = v0
        self.cmd_hdg = heading
        self.vel_step = 0
        self.hdg_step = 0
        self.vel_delta = 0
        self.hdg_delta = 0
    
    def update(self):
        vx = self.vel * cos(radians(90-self.hdg))
        vy = self.vel * sin(radians(90-self.hdg))
        self.x += vx
        self.y += vy
        
        if self.hdg_step > 0:
            self.hdg_step -= 1
            self.hdg += self.hdg_delta

        if self.vel_step > 0:
            self.vel_step -= 1
            self.vel += self.vel_delta
        return (self.x, self.y)

In [ ]:
def set_commanded_heading(self, hdg_degrees, steps):
    self.cmd_hdg = hdg_degrees
    self.hdg_delta = angle_between(self.cmd_hdg, 
                                   self.hdg) / steps
    if abs(self.hdg_delta) > 0:
        self.hdg_step = steps
    else:
        self.hdg_step = 0
     
def set_commanded_speed(self, speed, steps):
    self.cmd_vel = speed
    self.vel_delta = (self.cmd_vel - self.vel) / steps
    if abs(self.vel_delta) > 0:
        self.vel_step = steps
    else:
        self.vel_step = 0    


现在让我们实现一个带有噪声的模拟传感器。

In [ ]:
from numpy.random import randn

class NoisySensor(object):
    def __init__(self, std_noise=1.):
        self.std = std_noise

In [ ]:
def sense(self, pos):
        """传入实际位置作为元组（x，y）。
        返回添加noise的位置（x，y）"""
        
        return (pos [0] +（randn（）* self.std），
                pos [1] +（randn（）* self.std））

现在让我们生成一条轨迹并绘制它以测试一切是否正常工作。 我将把数据生成放在一个函数中，以便我们可以创建不同长度的路径（很快就会明白为什么）。

In [ ]:
import kf_book.book_plots as bp
import numpy as np
import matplotlib.pyplot as plt

def generate_data(steady_count, std):
    t = ManeuveringTarget(x0=0, y0=0, v0=0.3, heading=0)
    xs, ys = [], []

    for i in range(30):
        x, y = t.update()
        xs.append(x)
        ys.append(y)

    t.set_commanded_heading(310, 25)
    t.set_commanded_speed(1, 15)

    for i in range(steady_count):
        x, y = t.update()
        xs.append(x)
        ys.append(y)

ns = NoisySensor(std)
    pos = np.array(list(zip(xs, ys)))
    zs = np.array([ns.sense(p) for p in pos])
    return pos, zs

sensor_std = 2.
track, zs = generate_data(50, sensor_std)
plt.figure()
bp.plot_measurements(*zip(*zs), alpha=0.5)
plt.plot(*zip(*track), color='b', label='track')
plt.axis('equal')
plt.legend(loc=4)
bp.set_labels(title='轨迹 vs 测量值', x='X', y='Y')

这么多噪声让我们更容易看到各种设计选择的影响。

现在我们可以实现一个卡尔曼滤波器来跟踪这个对象。但是让我们做一个简化。*x*和*y*坐标是独立的，因此我们可以分别跟踪它们。在本章的其余部分中，我们将仅跟踪*x*坐标，以使代码和矩阵尽可能小。

我们从一个恒定速度滤波器开始。

In [ ]:
from filterpy.kalman import KalmanFilter
from filterpy.common import Q_discrete_white_noise

In [ ]:
def make_cv_filter(dt, std):
    cvfilter = KalmanFilter(dim_x = 2, dim_z=1)
    cvfilter.x = np.array([0., 0.])
    cvfilter.P *= 3
    cvfilter.R *= std**2
    cvfilter.F = np.array([[1, dt],
                           [0,  1]], dtype=float)
    cvfilter.H = np.array([[1, 0]], dtype=float)
    cvfilter.Q = Q_discrete_white_noise(dim=2, dt=dt, var=0.02)
    return cvfilter

def initialize_filter(kf, std_R=None):
    """ helper function - we will be reinitialing the filter
    many times.
    """
    kf.x.fill(0.)
    kf.P = np.eye(kf.dim_x) * .1
    if std_R is not None:
        kf.R = np.eye(kf.dim_z) * std_R

现在我们运行代码:

In [ ]:
sensor_std = 2.
dt = 0.1

# 初始化滤波器
cvfilter = make_cv_filter(dt, sensor_std)
initialize_filter(cvfilter)

track, zs = generate_data(50, sensor_std)

# 运行代码
z_xs = zs[:, 0]
kxs, _, _, _ = cvfilter.batch_filter(z_xs)

# 绘制结果
bp.plot_track(track[:, 0], dt=dt)
bp.plot_filter(kxs[:, 0], dt=dt, label='卡尔曼')
bp.set_labels(title='轨迹 vs 卡尔曼', x='时间（秒）', y='X');
plt.legend(loc=4);

In [ ]:
我们可以从图中看出，KalmanFilter无法跟踪航向的变化。从 **g-h Filter** 章节回忆起，这是因为滤波器没有建模加速度，因此它将始终滞后于输入。如果信号进入稳态，滤波器最终会赶上信号。让我们看看这个。

```python
# 重新初始化过滤器
dt = 0.1
initialize_filter(cvfilter)

track2, zs2 = generate_data(150, sensor_std)
xs2 = track2[:, 0]
z_xs2 = zs2[:, 0]

kxs2, _, _, _ = cvfilter.batch_filter(z_xs2)

bp.plot_track(xs2, dt=dt)
bp.plot_filter(kxs2[:, 0], dt=dt, label='卡尔曼')
plt.legend(loc=4)
bp.set_labels(title='加速度效应', 
              x='时间（秒）', y='X')

根本问题在于我们的过程模型对于稳态部分是正确的，但是在物体机动时是错误的。我们可以通过增加Q的大小来尝试解决这个问题，就像这样。

In [ ]:
# 重新初始化滤波器
dt = 0.1
initialize_filter(cvfilter)
cvfilter.Q = Q_discrete_white_noise(dim=2, dt=dt, var=2.0)
track, zs = generate_data(50, sensor_std)

# 重新计算轨迹
kxs2, _, _, _ = cvfilter.batch_filter(z_xs2)
bp.plot_track(xs2, dt=dt)
bp.plot_filter(kxs2[:, 0], dt=dt, label='KF')
plt.legend(loc=4)
bp.set_labels(title='大Q (var=2.0)', x='时间 (秒)', y='X')

我们可以看到滤波器更快地重新获取了轨迹，但代价是输出中有很多噪声。此外，许多跟踪情况无法容忍在第4秒和第8秒之间显示的延迟量。我们可以进一步减少它，但会以非常嘈杂的输出为代价，就像这样：

In [ ]:
# 重新初始化滤波器
dt = 0.1
initialize_filter(cvfilter)
cvfilter.Q = Q_discrete_white_noise(dim=2, dt=dt, var=50.0)
track, zs = generate_data(50, sensor_std)

# 重新计算轨迹
cvfilter.x.fill(0.)
kxs2, _, _, _ = cvfilter.batch_filter(z_xs2)

bp.plot_track(xs2, dt=dt)
bp.plot_filter(kxs2[:, 0], dt=dt, label='KF')
plt.legend(loc=4)
bp.set_labels(title='Huge Q (var=50.0)', x='time (sec)', y='X')

机动意味着加速，因此让我们实现一个恒定加速度的卡尔曼滤波器，并看看它在相同数据下的表现。

In [ ]:
def make_ca_filter(dt, std):
    cafilter = KalmanFilter(dim_x=3, dim_z=1)
    cafilter.x = np.array([0., 0., 0.])
    cafilter.P *= 3
    cafilter.R *= std
    cafilter.Q = Q_discrete_white_noise(dim=3, dt=dt, var=0.02)
    cafilter.F = np.array([[1, dt, 0.5*dt*dt],
                           [0, 1,         dt], 
                           [0, 0,          1]])
    cafilter.H = np.array([[1., 0, 0]])
    return cafilter

In [ ]:
def initialize_const_accel(f):
    f.x = np.array([0., 0., 0.])
    f.P = np.eye(3) * 3

In [ ]:
dt = 0.1
cafilter = make_ca_filter(dt, sensor_std)
initialize_const_accel(cafilter)

kxs2, _, _, _ = cafilter.batch_filter(z_xs2)

bp.plot_track(xs2, dt=dt)
bp.plot_filter(kxs2[:, 0], dt=dt, label='KF')
plt.legend(loc=4)
bp.set_labels(title='常加速度KalmanFilter',
              x='时间（秒）', y='X')

常加速度模型能够实时跟踪机动目标，但是代价是在稳态时期输出非常嘈杂。滤波器无法区分机动开始和信号噪声，因此信号噪声会被错误地认为是加速度，从而导致滤波器跟踪噪声。

似乎我们无法取得胜利。当目标加速时，恒定速度滤波器无法快速反应，但恒定加速度滤波器会将零加速度区间的噪声误解为加速度而不是噪声。

然而，这里有一个重要的见解将引导我们找到解决方案。当目标不机动时（加速度为零），恒定速度滤波器表现最佳。当目标机动时，恒定加速度滤波器表现良好，带有人工大过程噪声的恒定速度滤波器也表现良好。如果我们制作一个能够自适应跟踪对象行为的滤波器，我们就可以拥有最佳效果。

## 检测机动

在讨论如何创建自适应滤波器之前，我们必须问一下：“我们如何检测机动？”如果我们不知道机动何时发生，我们就无法合理地使滤波器适应机动。

我们一直将*机动*定义为跟踪对象加速的时间，但总的来说，如果对象的行为与滤波器使用的过程模型不同，我们可以说对象相对于卡尔曼滤波器是在机动。
机动对象对滤波器的数学影响是什么？对象的行为将与滤波器预测的不同，因此残差将很大。回想一下，残差是滤波器当前预测和测量之间的差异。
<img src="./figs/residual_chart.png">

为了确认这一点，让我们绘制滤波器在机动期间的残差。我将减少数据中的噪声，以便更容易看到残差。

In [ ]:
from kf_book.adaptive_internal import plot_track_and_residuals

以下是已翻译的文本：


def show_residual_chart():
    dt = 0.1
    sensor_std = 0.2

In [ ]:
# 初始化滤波器
cvfilter = make_cv_filter(dt, sensor_std)
initialize_filter(cvfilter)
pos2, zs2 = generate_data(150, sensor_std)
xs2 = pos2[:, 0]
z_xs2 = zs2[:, 0]

cvfilter.Q = Q_discrete_white_noise(dim=2, dt=dt, var=0.02)
xs, res = [], []
for z in z_xs2:
    cvfilter.predict()
    cvfilter.update([z])
    xs.append(cvfilter.x[0])
    res.append(cvfilter.y[0])
    
xs = np.asarray(xs)
plot_track_and_residuals(dt, xs, z_xs2, res)

show_residual_chart();

In [ ]:

在左边，我画出了嘈杂的测量结果与KalmanFilter输出之间的图像。在右边，我显示了滤波器计算出的residual - 测量值与KalmanFilter所做预测之间的差异。让我强调一下，以使其清晰明了。右侧的图不仅仅是左侧图中两条线之间的差异。左图显示的是测量值与最终KalmanFilter输出之间的差异，而右图向我们展示的是测量值与*过程模型的预测*之间的差异。

这可能看起来是微小的区别，但是从图中可以看出它并不是如此。当机动开始时，左图中的偏差量很小，但右图中的偏差告诉了我们一个不同的故事。如果跟踪的对象按照过程模型移动，residual图应该在0.0周围跳动。这是因为测量值将遵守方程式。

$$\mathtt{测量值} = \mathtt{过程模型}(t) + \mathtt{noise}(t)$$

一旦目标开始机动，目标行为的预测将不再与实际行为相匹配，因为方程式将变为

$$\mathtt{测量值} = \mathtt{过程模型}(t) + \mathtt{机动变化}(t) + \mathtt{noise}(t)$$

因此，如果residual偏离0.0的平均值，我们就知道机动已经开始。

从residual图中可以看出我们有很多工作要做。我们可以在residual图中清楚地看到机动的结果，但信号中的噪音量掩盖了机动的开始。这是我们提取信号的老问题。

## 可调节过程噪声

我们将考虑的第一种方法将使用较低阶模型，并根据是否发生机动来调整过程noise。当residual变得“大”时（对于某种合理的定义），我们将增加过程noise。这将使滤波器更倾向于测量而不是过程预测，并且滤波器将紧密跟踪信号。当residual很小时，我们将缩小过程noise。

在文献中有许多方法可以做到这一点，我将考虑几个选择。

### 连续调整

第一种方法（来自Bar-Shalom[1]）使用以下方程式对residual的平方进行归一化处理:

$$ \epsilon = \mathbf{y^\mathsf{T}S}^{-1}\mathbf{y}$$

其中$\mathbf{y}$是residual，$\mathbf{S}$是系统不确定性（协方差），其方程式为

$$\mathbf{S} = \mathbf{HPH^\mathsf{T}} + \mathbf{R}$$

如果用于计算的线性代数让您感到困惑，请回想一下我们可以将矩阵求逆看作除法，因此$\epsilon = \mathbf{y^\mathsf{T}S}^{-1}\mathbf{y}$可以看作是计算

$$\epsilon\approx\frac{\mathbf{y}^2}{\mathbf{S}}$$

$\mathbf{y}$和$\mathbf{S}$都是`filterpy.KalmanFilter`的属性，因此实现将会很简单。

让我们看一下$\epsilon$随时间的变化图。

```python
from numpy.linalg import inv
dt = 0.1
sensor_std = 0.2
cvfilter= make_cv_filter(dt, sensor_std)
_, zs2 = generate_data(150, sensor_std)

epss = []
for z in zs2[:, 0]:
    cvfilter.predict()
    cvfilter.update([z])
    y, S = cvfilter.y, cvfilter.S
    eps = y.T @ inv(S) @ y
    epss.append(eps)

t = np.arange(0, len(epss) * dt, dt)
plt.plot(t, epss)
bp.set_labels(title='Epsilon vs time', 
              x='time (sec)', y='$\epsilon$')

这个图表应该清晰地说明了归一化残差的效果。平方残差确保信号始终大于零，并且通过测量协方差进行归一化，使得我们可以区分残差相对于测量噪声明显变化的情况。机动开始于t=3秒，我们可以看到不久之后$\epsilon$开始迅速增加。

我们将会在 $\epsilon$ 超过某个限制时开始扩大 $\mathbf{Q}$，并在它再次降低到该限制以下时缩小。我们通过一个缩放因子来乘以 $\mathbf{Q}$。也许有关于分析选择这个因子的文献；我是通过实验得出的。在选择 $\epsilon$ 的限制值（命名为 $\epsilon_{max}$）方面，我们可以更加分析一些 - 一般来说，一旦残差大于3个标准差左右，我们可以假设这种差异是由于真正的变化而不是噪音引起的。然而，传感器很少是真正的高斯分布，因此在实践中使用较大的数字，例如5-6个标准差。

我已经使用合理的 $\epsilon_{max}$ 和 $\mathbf{Q}$ 缩放因子实现了此算法。为了更容易地检查结果，我将图表限制在模拟的前10秒钟。

In [ ]:
# 重新初始化滤波器
dt = 0.1
sensor_std = 0.2
cvfilter = make_cv_filter(dt, sensor_std)
_, zs2 = generate_data(180, sensor_std)

Q_scale_factor = 1000.
eps_max = 4.

xs, epss = [], []

count = 0
for i, z in zip(t, zs2[:, 0]):
    cvfilter.predict()
    cvfilter.update([z])
    y, S = cvfilter.y, cvfilter.S
    eps = y.T @ inv(S) @ y
    epss.append(eps)
    xs.append(cvfilter.x[0])

    if eps > eps_max:
        cvfilter.Q *= Q_scale_factor
        count += 1
    elif count > 0:
        cvfilter.Q /= Q_scale_factor
        count -= 1

bp.plot_measurements(zs2[:,0], dt=dt, label='z', alpha=0.5)
bp.plot_filter(t, xs, label='滤波器')
plt.legend(loc=4)
bp.set_labels(title='epsilon=4', x='time (sec)', y='$\epsilon$')

相比于常速滤波器，该滤波器的性能显着更好。常速滤波器需要大约10秒才能重新获取信号。而自适应滤波器在不到一秒钟的时间内就可以完成相同的操作。

### 连续调整 - 标准差版本

另一种非常相似的方法来自 Zarchan [2]，它基于测量误差协方差的标准差来设置限制。这里的方程式为：

$$ \begin{aligned}
std &= \sqrt{\mathbf{HPH}^\mathsf{T} + \mathbf{R}} \\
&= \sqrt{\mathbf{S}}
\end{aligned}
$$

如果残差的绝对值大于上述计算出的标准差的某个倍数，我们将增加一个固定量的过程噪声，重新计算 Q，并继续进行。

In [ ]:
from math import sqrt

def zarchan_adaptive_filter(Q_scale_factor, std_scale, 
                            std_title=False,
                            Q_title=False):
    cvfilter = make_cv_filter(dt, std=0.2)
    pos2, zs2 = generate_data(180-30, std=0.2)
    xs2 = pos2[:,0]
    z_xs2 = zs2[:,0]

    # 重新初始化滤波器
    initialize_filter(cvfilter)
    cvfilter.R = np.eye(1)*0.2

phi = 0.02
    cvfilter.Q = Q_discrete_white_noise(dim=2, dt=dt, var=phi)
    xs, ys = [], []
    count = 0
    for z in z_xs2:
        cvfilter.predict()  # 预测
        cvfilter.update([z])  # 更新
        y = cvfilter.y
        S = cvfilter.S
        std = sqrt(S)

In [ ]:
xs.append(cvfilter.x)
ys.append(y)

if abs(y[0]) > std_scale*std:
    phi += Q_scale_factor
    cvfilter.Q = Q_discrete_white_noise(2, dt, phi)
    count += 1
elif count > 0:
    phi -= Q_scale_factor
    cvfilter.Q = Q_discrete_white_noise(2, dt, phi)
    count -= 1

In [ ]:
xs = np.asarray(xs)
    plt.subplot(121)
    bp.plot_measurements(z_xs2, dt=dt, label='z')
    bp.plot_filter(xs[:, 0], dt=dt, lw=1.5)
    bp.set_labels(x='时间（秒）', y='$\epsilon$')
    plt.legend(loc=2)
    if std_title:
        plt.title(f'位置（std={std_scale}）')
    elif Q_title:
        plt.title(f'位置（Q scale={Q_scale_factor}）')
    else:
        plt.title('位置')
        
    plt.subplot(122)
    plt.plot(np.arange(0, len(xs)*dt, dt), xs[:, 1], lw=1.5)
    plt.xlabel('时间（秒）')
    if std_title:
        plt.title(f'速度（std={std_scale}）')
    elif Q_title:
        plt.title(f'速度（Q scale={Q_scale_factor}）')
    else:
        plt.title('速度')        
    plt.show()
 
zarchan_adaptive_filter(1000, 2, std_title=True)

所以我选择将噪声的缩放因子设为1000，标准差限制设为2。为什么是这些数字呢？嗯，首先，让我们看看2个标准差和3个标准差之间的差异。

**两个标准差**

In [ ]:
zarchan_adaptive_filter(1000, 2, std_title=True)

**三个标准差**

In [ ]:
zarchan_adaptive_filter(1000, 3, std_title=True)

我们可以从图表中看出，无论我们使用2个标准差还是3个标准差，位置的滤波器输出非常相似。但是速度的计算是一个不同的问题。让我们进一步探讨一下。首先，让我们将标准差设得非常小。

In [ ]:
zarchan_adaptive_filter(1000, .1, std_title=True)
zarchan_adaptive_filter(1000, 1, std_title=True)

随着标准差限制变小，速度计算变得更糟。思考一下为什么会这样。如果我们开始变化滤波器，以便在残差略微偏离预测时，它更喜欢测量，我们很快就会几乎全部将权重给予测量。没有预测的权重，我们就无法从中创建隐藏变量的信息。因此，当限制为0.1 std时，您可以看到速度被测量中的噪声淹没了。另一方面，由于我们非常偏爱测量，所以位置几乎完美地跟随机动。

现在让我们看看不同进程噪声增量的影响。在这里，我将标准差限制保持为2 std，并将增量从1变化到10,000。

In [ ]:
zarchan_adaptive_filter(1, 2, Q_title=True)
zarchan_adaptive_filter(10, 2, Q_title=True)
zarchan_adaptive_filter(100, 2, Q_title=True)
zarchan_adaptive_filter(1000, 2, Q_title=True)
zarchan_adaptive_filter(10000, 2, Q_title=True)

在这里我们可以看到随着增量因子的增加，位置估计略微变好，但速度估计开始产生大的超调。

我无法告诉您哪个是“正确”的。您需要测试滤波器对真实数据和模拟数据的性能，并选择最符合每个状态变量所需性能的设计。

## 衰减记忆滤波器

褪色记忆滤波器通常不被归类为自适应滤波器，因为它们不会自适应输入，但它们在机动目标方面提供良好的性能。它们还具有非常简单的计算形式，用于第一、二、三阶运动学滤波器（例如我们在本章中使用的滤波器）。这种简单形式不需要计算卡尔曼滤波器的增益的Ricatti方程，从而大大减少了计算量。然而，也有一种可与标准卡尔曼滤波器配合使用的形式。在本章中，我将重点介绍后者，因为我们的重点更多地放在自适应滤波器上。褪色记忆滤波器的这两种形式都在 `FilterPy` 中实现。

卡尔曼滤波器是递归的，但它将所有先前的测量值纳入到当前滤波增益的计算中。如果目标行为与过程模型一致，这就允许卡尔曼滤波器为每个测量找到最优估计。考虑一个正在飞行的球 - 如果我们考虑所有之前的测量，我们可以更清楚地估计球在时间t的位置。如果我们只使用了一些测量，我们对当前位置的确定性会更低，因此更容易受到测量噪声的影响。如果这仍然不清楚，请考虑最坏的情况。假设我们忘记了除最后一次测量和估计以外的所有内容。那么我们将对球的位置和轨迹没有信心，并且几乎别无选择，只能给当前测量分配更大的权重。如果测量存在噪声，则估计结果也会有噪声。每次初始化卡尔曼滤波器时都会出现这种效应。早期估计值存在噪声，但

随着获取更多的测量数据，它们会逐渐稳定下来。

然而，如果目标正在机动，它并不总是像过程模型预测的那样行为。在这种情况下，记住所有过去的测量和估计是一种负担。我们可以在上面的所有图表中看到这一点。目标启动转弯，而卡尔曼滤波器继续在直线上投影运动。这是因为滤波器已经建立了目标运动的历史记录，并错误地“感觉”到目标正在以给定的航向和速度直线运动。

衰减记忆滤波器通过给旧测量分配更小的权重，给最近的测量分配更大的权重来解决这个问题。

有许多衰减记忆滤波器的公式；我使用 Dan Simon 在 *Optimal State Estimation* [3] 中提供的公式。我不会进行推导，只提供结果。

卡尔曼滤波器估计误差的协方差方程为

$$ \bar{\mathbf P} = \mathbf{FPF}^\mathsf T + \mathbf Q $$

我们可以通过乘以一个项 $\alpha$ 来强制滤波器遗忘过去的测量值

$$ \tilde{\mathbf P} = \alpha^2\mathbf{FPF}^\mathsf T + \mathbf Q$$

其中 $\alpha > 1.0$。如果 $\alpha == 1$，则我们获得正常的卡尔曼滤波器性能。$\alpha$ 是 `KalmanFilter` 类的一个属性；它的值默认为 1，因此除非将 $\alpha$ 赋值为 1 以外的值，否则滤波器的行为就像卡尔曼滤波器。没有选择 $\alpha$ 的硬性规定，但通常非常接近于 1，例如 1.01。您需要使用模拟或实际数据进行多次运行，以确定对机动作出响应而不会由于过度加权嘈杂的测量而使估计变得过于嘈杂的值。

为什么这样做呢？如果我们增加估计误差协方差，则滤波器对其估计变得更加不确定，因此会给测量更多的权重。

一个警告 - 如果我们使用 $\alpha$，那么我们计算的是 $\tilde{\mathbf P}$，而不是 $\bar{\mathbf P}$。换句话说，`KalmanFilter.P` *不等于* 先验协方差，因此不要把它视为先验协方差。

让我们使用滤波器对我们的数据进行滤波，看看结果。我将向系统中注入大量误差，以便我们比较各种方法。

In [ ]:
pos2，zs2 = generate_data(70，std = 1.2)
xs2 = pos2 [:，0]
z_xs2 = zs2 [:，0]

cvfilter = make_cv_filter(dt，std = 1.2)
cvfilter.x.fill(0.)
cvfilter.Q = Q_discrete_white_noise(dim = 2，dt = dt，var = 0.02)
cvfilter.alpha = 1.00

xs，res = []，[]
for z in z_xs2:
    cvfilter.predict()
    cvfilter.update([z])
    xs.append(cvfilter.x[0])
    res.append(cvfilter.y[0])
xs = np.asarray(xs)
plt.subplot(221)
bp.plot_measurements(z_xs2，dt = dt，label = 'z')
plt.plot(t[0:100]，xs，label = 'filter')
plt.legend(loc = 2)
plt.title('标准KalmanFilter')

cvfilter.x.fill(0.)
cvfilter.Q = Q_discrete_white_noise(dim=2, dt=dt, var=20.)
cvfilter.alpha = 1.00

xs, res = [], []
for z in z_xs2:
    cvfilter.predict() # 预测
    cvfilter.update([z]) # 更新
    xs.append(cvfilter.x[0])
    res.append(cvfilter.y[0])
    
xs = np.asarray(xs)

plt.subplot(222)
bp.plot_measurements(z_xs2, dt=dt, label='z') # 绘制 z 数据
plt.plot(t[0:100], xs, label='filter') # 绘制滤波器数据
plt.legend(loc=2)
plt.title('$\mathbf{Q}=20$')

cvfilter.x.fill(0.)
cvfilter.Q = Q_discrete_white_noise(dim=2, dt=dt, var=0.02)
cvfilter.alpha = 1.02

xs, res = [], []
for z in z_xs2:
    cvfilter.predict()
    cvfilter.update([z])
    xs.append(cvfilter.x[0])
    res.append(cvfilter.y[0])
xs = np.asarray(xs)
plt.subplot(223)
bp.plot_measurements(z_xs2, dt=dt, label='z')
plt.plot(t[0:100], xs, label='filter')
plt.legend(loc=2)
plt.title('衰减记忆 ($\\alpha$ = 1.02)')

cvfilter.x.fill(0.)
cvfilter.Q = Q_discrete_white_noise(dim=2, dt=dt, var=0.02)
cvfilter.alpha = 1.05

xs，res = []，[]
对于 z_xs2 中的每个 z：
    cvfilter.predict()
    cvfilter.update([z])
    xs.append(cvfilter.x [0])
    res.append(cvfilter.y [0])
xs = np.asarray(xs)
plt.subplot(224)
bp.plot_measurements(z_xs2，dt = dt，label = 'z')
plt.plot(t [0:100]，xs，label = 'filter')
plt.legend(loc = 2)
plt.title('Fading Memory ($\\alpha$ = 1.05)');```


第一张图显示了卡尔曼滤波器的性能。当机动开始时，滤波器发散，并且直到大约10秒后才重新获得信号。然后，我通过使过程噪声较大来迅速使滤波器跟踪机动，但这会使滤波器估计非常嘈杂，因为会非常重视嘈杂的测量值。然后，我使用 $\alpha=1.02$ 实现了一个衰减记忆滤波器。过滤后的估计非常平滑，但当目标恢复稳态行为时，需要几秒钟才能收敛。但是，它所需的时间比卡尔曼滤波器要小得多，滞后量也小得多-衰减记忆的估计比卡尔曼滤波器的估计更接近实际轨迹。最后，我将 $\alpha$ 提高到1.05。在这里，我们可以看到滤波器对机动的响应几乎是即时的，但是在稳态操作期间，估计值不太直，因为滤波器在遗忘。

以往的测量记录。

这是代码修改所带来的很好的性能！请注意这里没有“正确”的选择。您需要根据您的需求以及测量噪声、过程噪声和目标机动行为的特性来设计您的滤波器。

## 多模型估计

本章中我一直在使用一个目标在稳态下移动、执行机动，然后返回稳态的示例。我们将其视为两种模型——恒定速度模型和恒定加速度模型。每当您可以将系统描述为服从有限一组模型中的一个时，您可以使用*多模型（MM）估计*。我们使用一系列多个滤波器，每个滤波器使用不同的过程来描述系统，并根据所跟踪对象的动态情况在它们之间切换或混合。

正如你所想象的那样，这是一个广泛的话题，有许多设计和实现MM估计器的方法。但考虑本章追踪的目标的简单方法。其中一个想法是同时运行一个恒定速度和恒定加速度滤波器，并在检测到机动时通过检查残差来在它们的输出之间切换。即使这个选择也给了我们很多选择。考虑一个转向物体的动力学。例如，汽车在车轮上转弯 - 前轮转向，汽车围绕后轮旋转。这是一个非线性过程，因此为了获得最佳结果，我们需要使用某种类型的非线性滤波器（EKF、UKF等）来模拟转弯。另一方面，线性恒定速度滤波器在旅行的稳态部分表现良好。因此，我们的滤波器库可能包括线性KF和转弯的EKF滤波器。但是，两者都不是特别适合建模行为，例如

加速和刹车。因此，一个高性能的 MM 估计器可能包含许多滤波器组成的银行，每个滤波器都设计为针对跟踪对象的某个性能范围表现最佳。

当然，您不需要根据模型的顺序来设置滤波器。您可以在每个滤波器中使用不同的噪声模型和适配器。例如，在上面的部分中，我展示了许多图表，描述了改变参数对速度和位置估计的影响。也许某种设置对位置更有效，而对速度的不同设置更有效。将两者放入您的滤波器库中。然后，您可以从一个滤波器中获取最佳位置估计，从另一个滤波器中获取最佳速度估计。

### 两个滤波器自适应滤波器

我相信切换滤波器以获得最佳性能的想法是清晰的，但我们应该使用什么数学基础来实现它呢？我们面临的问题是，尝试通过嘈杂的测量来检测何时应该将制度变化转化为模型变化。卡尔曼滤波器的哪个方面测量测量值与预测值偏差的程度？是*残差*。

假设我们有一个一阶（恒定速度）的卡尔曼滤波器。只要目标不进行机动，滤波器就会紧密跟踪其行为，并且大约68%的测量值应在1$\sigma$内。此外，残差应该围绕0波动，因为如果传感器是高斯的，那么正误差与负误差的数量应该相等。如果残差增长并且超出了预测的范围，则目标必须不像状态模型所预测的那样执行。我们在此图表中早些时候看到了这一点，当跟踪对象开始机动时，残差从围绕0反弹到突然跳跃并保持在零上方。

In [ ]:
show_residual_chart()

对于这个问题，我们发现在目标处于稳态时，恒定速度滤波器的表现要优于恒定加速度滤波器，而当目标机动时则相反。在上面的图表中，这种转变发生在4秒钟处。

因此，该算法很简单。初始化恒定速度和恒定加速度滤波器，并在预测/更新循环中一起运行。每次更新后，检查恒定速度滤波器的残差。如果在理论范围内，则使用恒定速度滤波器的估计值作为估计值，否则使用恒定加速度滤波器的估计值。

In [ ]:
def run_filter_bank(threshold, show_zs=True):
    dt = 0.1
    cvfilter= make_cv_filter(dt, std=0.8) # 创建恒定速度滤波器
    cafilter = make_ca_filter(dt, std=0.8) # 创建恒定加速度滤波器
    pos, zs = generate_data(120, std=0.8) # 生成data
    z_xs = zs[:, 0]
    xs, res = [], []

In [ ]:
for z in z_xs:
        cvfilter.predict()
        cafilter.predict()
        cvfilter.update([z])
        cafilter.update([z])
        
        std = np.sqrt(cvfilter.R[0,0])
        if abs(cvfilter.y[0]) < 2 * std:
            xs.append(cvfilter.x[0])
        else:
            xs.append(cafilter.x[0])
        res.append(cvfilter.y[0])
    xs = np.asarray(xs)
    if show_zs:
        plot_track_and_residuals(dt, xs, z_xs, res)
    else:
        plot_track_and_residuals(dt, xs, None, res)

run_filter_bank(threshold=1.4)

在这里，滤波器紧密跟踪机动。当目标没有机动时，我们的估计几乎没有噪声，然后一旦它开始机动，我们就会快速检测到并切换到恒定加速度滤波器。但是，这不是理想的。这是滤波器的单独输出图：

In [ ]:
run_filter_bank(threshold=1.4, show_zs=False)

你可以看到，当滤波器切换时，估计值会出现跳跃。我不会在生产系统中使用这个算法。接下来的部分介绍了一种消除此问题的滤波器组的最新实现。

## MMAE

使用几个滤波器来检测机动的核心思想是合理的，但是当我们突然在滤波器之间转换时，估计值会出现不规则波动。在这本书中使用概率来确定测量和模型的“可能性”，因此我们不会选择“测量”或“预测”，具体取决于哪个更有可能，而是按照它们的可能性比例选择两者的“混合物”。我们在这里也应该采用同样的方法。这种方法称为*多模型自适应估计器*（MMAE）。

在**设计卡尔曼滤波器**章节中，我们学习了*可能性函数*。

$$\mathcal{L} = \frac{1}{\sqrt{2\pi S}}\exp [-\frac{1}{2}\mathbf{y}^\mathsf{T}\mathbf{S}^{-1}\mathbf{y}]$$

这告诉我们一个滤波器在给定输入时能够最优地执行的可能性。$\mathbf y$ 是残差，$\mathbf S$ 是系统不确定性（测量空间中的协方差）。这只是残差和系统不确定性的高斯分布。大的残差将会导致大的不确定性，因此低的可能性会导致测量与滤波器当前状态不匹配。我们可以使用此方法计算每个滤波器是最佳拟合数据的概率。如果我们有N个滤波器，我们可以计算滤波器i相对于其余滤波器的正确概率

$$p_k^i = \frac{\mathcal{L}_k^ip_{k-1}^i}{\sum\limits_{j=1}^N \mathcal{L}_k^jp_{k-1}^j}$$

看上去有点乱，但很简单。分子只是这个时间步长的似然乘以上一时间帧此滤波器正确的概率。我们需要所有滤波器的概率之和为1，因此我们通过除以分母中所有其他滤波器的概率来进行归一化。

这是一个递归定义，因此我们需要为每个滤波器分配一些初始概率。在没有更好的信息的情况下，为每个滤波器使用 $\frac{1}{N}$。然后，我们可以将每个滤波器的状态乘以其正确性的概率，再将它们相加，从而计算出估计状态。

这是一个完整的实现：

In [ ]:
def run_filter_bank():
    dt = 0.1
    cvfilter = make_cv_filter(dt, std=0.2)
    cafilter = make_ca_filter(dt, std=0.2)

    _, zs = generate_data(120, std=0.2)
    z_xs = zs[:, 0]
    xs, probs = [], []

In [ ]:
pv，pa = 0.8，0.2
    pvsum，pasum = 0.，0.
    
    for z in z_xs:
        cvfilter.predict() # 预测
        cafilter.predict()
        cvfilter.update([z]) # 更新
        cafilter.update([z])
        
        cv_likelihood = cvfilter.likelihood * pv
        ca_likelihood = cafilter.likelihood * pa
        
        pv = (cv_likelihood) / (cv_likelihood + ca_likelihood)
        pa = (ca_likelihood) / (cv_likelihood + ca_likelihood)
        
        x = (pv * cvfilter.x[0]) + (pa*cafilter.x[0])
        xs.append(x)
        probs.append(pv / pa)

    xs = np.asarray(xs)
    t = np.arange(0, len(xs) * dt, dt)
    plt.subplot(121)
    plt.plot(t, xs) # 左图为滤波器估计值，以便查看结果的平滑程度。
    plt.subplot(122)
    plt.plot(t, xs) # 右图为估计值和测量值，以证明滤波器正在跟踪机动。
    plt.plot(t, z_xs)
    return xs, probs

xs, probs = run_filter_bank()

再次强调，这仅仅是我们在整本书中一直在使用的贝叶斯算法。我们有两个（或更多）测量或估计值，并分别附带一个概率值。我们选择一个估计值作为这些值的加权组合，其中权重与正确概率成比例。每一步概率的计算公式为

$$\frac{\texttt{Prob(meas | state)} \times\texttt{prior}}{\texttt{normalization}}$$

这就是贝叶斯定理。

对于实际问题，你可能需要在你的滤波器组中使用两个以上的滤波器。在我的工作中，我使用计算机视觉来跟踪物体。我跟踪曲棍球。曲棍球会滑动，弹跳和滑动，滚动，弹开，被拾起和携带，并且被球员快速地“运球”。我跟踪运动员，他们的非线性行为能力几乎是无限的。在这些情况下，两个滤波器组根本就无济于事。我需要建模多个过程模型，不同的假设噪声，由计算机视觉检测引起的等等。但你已经得到了主要的想法。

### MMAE滤波器的局限性

我所介绍的MMAE有一个重大问题。看看这个常速度与常加速度滤波器概率比的图表。

In [ ]:
plt.plot(t[0:len(probs)], probs)
plt.title('probability ratio p(cv)/p(ca)')
plt.xlabel('time (sec)');

在前三秒钟，被跟踪的物体沿直线行驶时，恒定速度滤波器比恒定加速度滤波器更有可能。一旦机动开始，概率很快改变，倾向于恒定加速度模型。但是，机动在第6秒完成。你可能会期望恒定速度滤波器的概率再次变大，但实际上它仍然保持在零。

这是由于概率的递归计算：

$$p_k = \frac{\mathcal{L}p_{k-1}}{\sum \text{probabilities}}$$

一旦概率变得非常小，它就再也无法恢复。结果是，滤波器组迅速收敛于仅最有可能的滤波器。一个强大的方案需要监视每个滤波器的概率，并杀死那些概率非常低的滤波器，并用表现更好的滤波器替换它们。您可以将现有的滤波器细分为新滤波器，试图跨越使它们表现良好的特性。在最坏的情况下，如果一个滤波器发散了，您可以重新初始化滤波器的状态，使其更接近当前测量值。

## 交互多模型（IMM）

让我们从另一个角度考虑多个模型。场景与之前相同-我们希望跟踪机动目标。我们可以设计一组卡尔曼滤波器，它们做出不同的建模假设。它们可以在滤波器顺序或过程模型中的噪声量方面有所不同。每当有新的测量结果出现时，每个滤波器都有成为正确模型的概率。

这种天真的方法会导致组合爆炸。在第1步中，我们生成$N$个假设，即每个滤波器一个。在第2步中，我们生成另外$N$个假设，然后需要将其与先前的$N$个假设结合起来，这将产生$N^2$个假设。已经尝试了许多不同的方案，其中一些方案会剔除不太可能的假设或合并相似的假设，但这些算法仍然面临计算成本高和/或性能差的问题。我不会在本书中涵盖这些内容，但是在文献中有着著名的广义伪贝叶斯（GPB）算法的例子。

*互动多模型*（IMM）算法是由Blom[5]发明的，用于解决多模型的组合爆炸问题。由Blom和Bar-Shalom撰写的随后的一篇论文是最被引用的论文[6]。其思想是针对系统可能的每种行为模式都有一个滤波器。在每个时期，我们让滤波器*相互作用*。较可能的滤波器修改较不可能的滤波器的估计，以使它们更接近于表示当前系统状态的状态。这种混合是以概率的形式完成的，因此不太可能的滤波器也会以较小的幅度修改可能的滤波器。

例如，假设我们有两种模式：直行或转弯。每种模式都通过一个Kalman滤波器表示，可能是一个一阶和二阶滤波器。现在假设目标正在转弯。第二阶滤波器将产生一个良好的估计，而第一阶滤波器将滞后信号。每个的似然函数告诉我们哪个滤波器最有可能。第一阶滤波器的似然性很低，因此我们用第二阶滤波器大大调整其估计。第二阶滤波器非常可能，因此第一阶Kalman滤波器只会稍微改变其估计。

现在假设目标停止旋转。由于我们一直在使用二阶估计修正一阶滤波器的估计，所以它不会滞后信号太多。在仅仅几个周期内，它将产生非常好的（高概率的）估计，并成为最可能的滤波器。然后它将开始大量贡献于二阶滤波器的估计。回想一下，二阶滤波器将测量噪声误认为是加速度。这个调整可以大大减少这种影响。

### 模式概率

我们为系统定义了一组模式$m$，并假设目标始终处于其中一个模式中。在上面的讨论中，我们有直行和旋转两种模式，因此$m=\{\text{直行},\ \text{旋转}\}$。

我们为目标出现在任何给定模式中分配一个概率。这给我们一个包含每个可能模式的一个概率的向量，称为*模式概率*。$m$ 有两种模式，因此我们将有一个包含两个概率的向量。如果我们认为目标有 70% 的几率直行，我们可以表示为

$$\mu = \begin{bmatrix} 0.7 & 0.3\end{bmatrix}$$

我们在转弯的概率上得到了 0.3，因为概率必须加起来等于一。通常但不普遍地，$\mu$ 被用作模式概率的符号，所以我会使用它。请不要将其与平均值混淆。

在 Python 中，我们可以这样实现:

In [ ]:
mu = np.array([0.7, 0.3])
mu

我们可以通过以下方式将其形式化，即假设在先前的测量 $Z$ 给定的情况下，$m_i$ 正确的先验概率（机动物体处于模式 $i$）为

$$\mu_i = P(m_i|Z)$$

### 模式转换

接下来我们需要考虑这是一个机动目标。它会直线行驶，然后转弯，接着又直线行驶。我们可以将这些模式之间的转换建模为 [*马尔可夫链*](https://zh.wikipedia.org/wiki/%E9%A9%AC%E5%B0%94%E5%8F%AF%E5%A4%AB%E9%93%BE)，如下图所示：

In [ ]:
import kf_book.adaptive_internal as adaptive_internal
adaptive_internal.plot_markov_chain()

这显示了一个目标的两种模式的示例，即直线行驶和执行转弯。如果目标的当前模式是直线行驶，则我们预测有97%的概率目标会继续直线行驶，有3%的概率会开始转弯。一旦目标开始转弯，我们预测有95%的概率会继续转弯，有5%的概率会返回直线路径。

该算法对于确切的数字不敏感，通常您会使用模拟或试验来选择适当的值。然而，这些值是相当有代表性的。

我们使用[*转移概率矩阵*](https://en.wikipedia.org/wiki/Stochastic_matrix)来表示马尔可夫链，我们将其称为$\mathbf M$。对于插图中的马尔可夫链，我们将写作

$$\mathbf M = \begin{bmatrix}.97 & .03\\.05 & .95\end{bmatrix}$$

换句话说，$\mathbf M [i，j]$ 是上一个状态是 $i$ 的情况下，状态为 $j$ 的概率。在这个例子中，当前状态为直行$(j = 0)$，上一个状态为转弯$(i = 1)$ 的概率是 $\mathbf M [1，\ 0] = 0.05$。在Python中，我们会写成：

In [ ]:
M = np.array([[.97, .03], [.05, .95]])
print(M)
print('从转弯到直行的概率是', M[1, 0], '百分比')

这使我们能够根据转移概率计算新的模态概率。 让我们计算转移后模式为直行的概率。 我们有两种方法可以直行。 我们可以一直直行，或者我们可以转弯，然后直行。 前者的概率是 $(0.7\times 0.97)$，后者的概率是 $(0.3\times 0.05)$。 我们将模态概率与马尔科夫链中相关的概率相乘。 *总概率* 是两者之和，即 $(0.7)(0.97) + (0.3)(0.05) = 0.694$。

回想第二章的[*总概率定理*](https://en.wikipedia.org/wiki/Law_of_total_probability)。 它指出几个不同事件的概率为

$$P(A) = \sum P(A\mid B)\, P(B)$$

这里$P(A\mid B)$是转移矩阵$\mathbf M$，$P(B)$是$\mu$。我们使用数组和矩阵，利用向量乘以矩阵的事实来计算乘积之和：

$$\begin{bmatrix}\mu_1 & \mu_2 \end{bmatrix}\begin{bmatrix}m_{11} & m_{12}\\m_{21} & m_{22}\end{bmatrix} = \begin{bmatrix}\mu_1 m_{11} + \mu_2 m_{21} & \mu_1 m_{12} + \mu_2 m_{22}\end{bmatrix}$$

IMM文献将此表示为

$$\bar c_j = \sum\limits_{i=1}^{N} \mu_i M_{ij}$$

我们使用NumPy的`dot`函数来计算。我们也可以使用矩阵乘法运算符`@`，但我发现使用点号（`dot`函数）作为求和符号更直观：

In [ ]:
cbar = np.dot(mu, M)
cbar

### 计算模式概率

我们将使用贝叶斯定理计算新的模式概率。回想一下贝叶斯定理的陈述：

$$\text{后验概率} = \frac{\text{先验概率} \cdot \text{似然}}{\text{归一化因子}}$$

这里的先验概率是我们在上一节中计算的总概率。卡尔曼滤波器计算“似然度”，即在给定滤波器当前状态的情况下测量值的似然度。回顾一下方程式：

$$
\mathcal{L} = \frac{1}{\sqrt{2\pi \mathbf S}}\exp [-\frac{1}{2}\mathbf y^\mathsf T\mathbf S^{-1}\mathbf y]$$

在数学符号中，更新后的模态概率为：

$$\mu_i = \| \mathcal{L}_i {\bar c}_{i}\|$$

换句话说，对于每个Kalman滤波器（模式），我们将模式概率计算为考虑可能的转换时当前模式的概率与这是正确模式的似然度的乘积。然后，我们规范化所有概率，使它们总和为一。

这在Python中很容易计算。我将介绍变量`L`以存储似然值。似然值由`KalmanFilter.update()`步骤计算，在下面的代码片段中，由于我们还没有创建卡尔曼滤波器，我只是硬编码了`L`的值：

In [ ]:
# L = [kf0.L, kf1.L]  # get likelihoods from Kalman filters
L = [0.000134，0.0000748] 
mu = cbar * L
mu /= sum(mu) # normalize
mu

在这里，您可以看到直线滤波器相对较强的似然性将直线模式的概率从70%推到了80.2％。

## 混合概率

此时，我们可以使用模式转换来计算所有可能选择的概率。如果$\mu = \begin{bmatrix} 0.63 & 0.27\end{bmatrix}$，那么我们可以使用转换概率矩阵来计算所有可能的结果。换句话说，如果当前模式是直行$(\mu=0.63)$，我们可以根据目标是继续直行还是转弯来计算出两个新的概率。我们对转弯模式$(\mu=0.27)$做同样的事情。我们将从2个模式概率变成4个。在下一步中，4个将变成8个，以此类推。这是计算上的精确，但在实践中不可行。只有30个时期后，您将需要8GB的内存以双精度存储模式概率。

我们需要更好的、尽管是近似的方法。IMM 通过计算*混合概率*来解决这个问题。这个想法很简单。假设第一种模式（直行）当前非常可能，而第二种模式（转弯）则不太可能。我们不再让卡尔曼滤波器以所有滤波器的加权平均值计算直行模式的状态，而是让与目标模式匹配可能性更高的滤波器比可能性较低的滤波器得到更高的加权。结果是概率较高的滤波器的信息提高了不太可能的滤波器的准确性。这是算法的关键。

我们需要做的非常简单。每个卡尔曼滤波器执行更新步骤，计算出新的均值和协方差。然后，根据我们称之为*混合概率*的加权和计算每个滤波器的新均值和协方差。可能的滤波器将被不太可能的滤波器略微调整，而不太可能的滤波器将被可能的滤波器强烈调整。文献将这些调整后的均值和协方差称为*混合条件*或*混合初始条件*。我使用符号$\mathbf x^m_j$表示混合状态，$\mathbf P^m_j$表示混合协方差。方程式为：

$$\begin{aligned}
\mathbf x^m_j &= \sum_{i=1}^N \omega_{ij} \mathbf x_i \\
\mathbf P^m_j &= \sum_{i=1}^N \omega_{ij}\left[(\mathbf x^i - \mathbf  x^m_i) (\mathbf x^i - \mathbf  x^m_i)^\mathsf T + \mathbf P_i\right]
\end{aligned}$$

将下标视为数组的索引即可。以伪Python代码表示如下：

In [ ]:
for j in N:
    x0[j] = sum_over_i(w[i,j] * x[i])
    P0[j] = sum_over_i(w[i, j] * (P[i] + np.outer(x[i] - x0[j]))) 

不要让符号混淆了一个简单的想法：将来自可能滤波器的估计值合并到来自不可能滤波器的估计值中，确保所有滤波器都有一个良好的估计值。

我们如何计算混合概率？想一想，在继续阅读之前尝试给出一个合理的答案。我们有描述每种模式的当前概率的模式概率，然后有描述我们更改模式的可能性的转换概率。我们如何计算新的概率？

当然是贝叶斯定理！ 先验乘以似然，再进行归一化。先验是模式概率，似然来自马尔可夫链，我们将其存储在矩阵$\mathbf M$中。

$$\boldsymbol\omega_{ij} = \| \mu_i \cdot \mathbf M_{ij}\|$$

我们可以按如下方式计算。我之前错乱地计算了$\mu$和$\bar c$的更新（你必须将转移概率矩阵纳入$\mu$中计算$\bar c$），因此我需要在这里进行更正：

In [ ]:
cbar = np.dot(mu, M) #计算目标在模式j中的总概率

omega = np.zeros((2, 2))
for i in range(2):
    for j in range(2):
        omega[i, j] = (M[i, j] * mu[i]) / cbar[j]
omega

Kalman滤波器需要执行预测步骤来计算新的先验概率。 它们使用混合估计值：

$$
\begin{aligned}
\bar{\mathbf x}_j &= \mathbf F_j\mathbf x^m_j\\
\bar{\mathbf P}_j &= \mathbf F_j\mathbf P^m_j\mathbf F_j^\mathsf T + \mathbf Q_j
\end{aligned}$$

### IMM估计

现在，我们需要从滤波器组中得出最终状态估计值。我们如何做到这一点？只需对每个Kalman滤波器的混合估计进行加权即可：

$$\begin{aligned}
\mathbf x &= \sum_{j=1}^N \mu_j{\bar{\mathbf x}}_j\\
\mathbf P &= \sum_{j=1}^N \mu_j\left[(\bar{{\mathbf x}}_j - \bar{\mathbf x})({\bar{\mathbf x}}_j - \bar{\mathbf x})^\mathsf T + \bar{\mathbf P_j}\right]
\end{aligned}$$

### 使用IMM跟踪机动目标

我们来看一个例子。Crassidis[4]是少数几本具有实例的书籍之一，因此我选择了他的例子。他追踪一个移动目标600秒。目标一开始直线运动，然后在400秒时注入控制输入，使目标转向90度。他使用两个恒定加速度的卡尔曼滤波器。一个滤波器假定没有过程噪声，另一个假定具有谱密度为$10^{-3}\mathbf I$的过程噪声。他假设滤波器非常好地初始化，将$\mathbf P$设为$10^{-12}$。我的实现如下：

In [ ]:
import copy
from scipy.linalg import block_diag
from filterpy.kalman import IMMEstimator
from filterpy.common import Saver

N = 600
dt = 1.
imm_track = adaptive_internal.turning_target(N)

# 创建带噪声的测量值
zs = np.zeros((N, 2))
r = 1
for i in range(N):
    zs[i, 0] = imm_track[i, 0] + randn()*r
    zs[i, 1] = imm_track[i, 2] + randn()*r

ca = KalmanFilter(6, 2)
dt2 = (dt**2)/2
F = np.array([[1, dt, dt2],
              [0,  1,  dt],
              [0,  0,   1]])
            
ca.F = block_diag(F, F)
ca.x = np.array([[2000., 0, 0, 10000, -15, 0]]).T
ca.P *= 1.e-12
ca.R *= r**2
q = np.array([[.05, .125, 1/6],
              [.125, 1/3, .5],
              [1/6, .5, 1]])*1.e-3
ca.Q = block_diag(q, q)
ca.H = np.array([[1, 0, 0, 0, 0, 0],
                 [0, 0, 0, 1, 0, 0]])

# 创建与上一个滤波器相同的滤波器，但不包含过程误差
cano = copy.deepcopy(ca)
cano.Q *= 0

filters = [ca, cano]

In [ ]:
M = np.array([[0.97, 0.03],
              [0.03, 0.97]])
mu = np.array([0.5, 0.5])
bank = IMMEstimator(filters, mu, M)

xs, probs = [], []
for i, z in enumerate(zs):
    z = np.array([z]).T
    bank.predict()
    bank.update(z)

    xs.append(bank.x.copy())
    probs.append(bank.mu.copy())

xs = np.array(xs)
probs = np.array(probs)
plt.subplot(121)
plt.plot(xs[:, 0], xs[:, 3], 'k')
plt.scatter(zs[:, 0], zs[:, 1], marker='+')

plt.subplot(122)
plt.plot(probs[:, 0])
plt.plot(probs[:, 1])
plt.ylim(-1.5, 1.5)
plt.title('概率比 p(cv)/p(ca)');


很难看出滤波器的性能，因此让我们仅查看转弯开始时的性能。我交换了 $x$ 和 $y$ 轴，以便让我们可以近距离观察。在下图中，转弯开始于 $Y=4000$。如果你非常仔细地看，你会发现在启动转弯后估计略微波动，但滤波器能够及时跟踪测量并很快平稳地跟踪。

In [ ]:
plt.plot(xs[390:450, 3], xs[390:450, 0], 'k')
plt.scatter(zs[390:450, 1], zs[390:450, 0], marker='+', s=100);
plt.xlabel('Y'); plt.ylabel('X')
plt.gca().invert_xaxis()
plt.axis('equal');

### IMM的局限性

我还没有广泛地使用IMM，所以无法像我希望的那样详细介绍它。 然而，IMM的发明是为了跟踪机动飞行器的空中交通管制，并且根据所有报告，它在该角色中表现出色。

这种使用情况假定了一些事情。 其中最重要的是银行中所有滤波器都具有相同的维度设计要求。 对数学的回顾应该可以说明原因。 为了创建混合估计，IMM执行以下计算：

$$\mathbf x = \sum_{j=1}^N \mu_j{\bar{\mathbf x}}_j$$

只有在每个滤波器中的状态$x$具有相同的维度时，才能计算出这个值。 此外，$x[i]$的解释对于每个滤波器都必须相同。

例如，假设我们尝试使用一个使用恒定速度模型的滤波器和另一个使用恒定加速度模型的滤波器。这不起作用，因为$x$的维度不同。如果您尝试使用具有不同维度的滤波器，则`FilterPy`会引发`ValueError`。

In [ ]:
ca = KalmanFilter(3, 1)
cv = KalmanFilter(2, 1)

trans = np.array([[0.97, 0.03],
                  [0.03, 0.97]])

imm = IMMEstimator([ca, cv], (0.5, 0.5), trans)

我偶尔会收到有关此问题的电子邮件或错误报告。过去，我建议将具有恒定速度的滤波器设计为3维，并实现 `F` 忽略加速度。

In [ ]:
F = np.array([[1, dt, 0],
              [0, 1,  0],
              [0, 0,  0]])

回过头来看，我不确定这是否是一个合理的建议。它允许IMM工作，但显然加速度的估计值将是不正确的，因为其中一个滤波器将具有加速度的准确估计，而另一个滤波器将具有0的估计值。这种不准确的加速度将被用于执行下一个预测周期。

考虑一个更极端的情况。假设你的其中一个滤波器将 `x[2]` 解释为加速度，而另一个滤波器将其解释为角速度。显然，混合估计值的 `x[2]` 将毫无意义，因为你不能将（线性）加速度与旋转速率相加。

正如我所说，我对IMM并不是特别了解。也许文献中解释了如何处理这些情况。我只能说，FilterPy实现的IMM不适用于这些用例。

IMM被设计用于空中交通管制，使用具有不同过程假设的滤波器。一架飞机可以飞行平稳，可以上升/下降，可以进行协调转弯或非协调转弯。您可以为每种情况设计一个具有不同`F`和`Q`矩阵的滤波器，但状态估计`x`对于所有情况都是相同的。

## 摘要

本章包含了本书中一些更具挑战性的材料。然而，它是实现现实卡尔曼滤波器的入口。如果我们正在控制一个机器人，我们知道它的过程模型，很容易为它构建卡尔曼滤波器。然而，更常见的情况是给我们一组时间序列数据，并要求我们理解它。我们的过程模型在很大程度上是未知的。我们使用本章中的技术来“学习”（在机器学习意义上）如何参数化我们的模型。目标机动变化，因此我们的滤波器必须是自适应的。

寻找最优答案涉及组合爆炸，实际上是不切实际的。由于其良好的性能和计算可行性，IMM算法已成为标准算法。

一个真正的滤波器组通常包含两个以上的滤波器。往往会有很多滤波器。随着目标状态的改变，一些滤波器变得微不足道。大多数自适应滤波器实现了一种算法，以消除极不可能的滤波器，并用更符合当前状态的滤波器替换它们。这是高度特定于您的问题空间，通常非常特定。您需要设计杀死和创建滤波器的方案，并根据模拟或实际数据对其进行测试。

尽管算法很复杂，但我希望你能认识到其中的基本思想非常简单。我们使用了在第二章学到的相同的两个工具：贝叶斯定理和全概率定理。我们使用贝叶斯定理来整合新的信息，并使用全概率定理计算过程模型的影响。

对我而言，这一章强调了卡尔曼滤波器贝叶斯公式的美妙之处。我并不在意您是否学会了IMM算法的细节。我希望您能看到，非常简单的概率推理导致了这些结果。卡尔曼博士推导的卡尔曼滤波器的线性代数方程来自一种不同的推理方式，称为*正交投影*。它以其自己的方式很美丽，我敦促您阅读他的论文。但我不确定我是否发现它们易于使用，也不清楚如何使用这些技术设计新的非最优滤波器，比如IMM。相比之下，贝叶斯定理让我们轻松处理这些问题。

## 参考资料

 * [1] Bar-Shalom, Y., Xiao-Rong L., and Thiagalingam Kirubarajan. *应用于跟踪和导航的估计方法*。纽约：Wiley，第424页，2001年。

* [2] Zarchan, P., and Musoff, H., *Kalman滤波的基础：实用方法* 第四版. 美国航空航天学会, p.584-595, 2015. 


* [3] Simon, D., *最优状态估计：Kalman、H和非线性方法*，Hoboken, NJ: Wiley-Interscience, p. 208-212, 2006


* [4] Crassidis, John L.和John L. Junkins. *动态系统的最优估计*。CRC出版社，2011年。


* [5] Blom, H.A.P.，“突变系统的高效滤波器”，*第23届决策与控制会议文集*，拉斯维加斯，内华达州，1984年12月。


* [6] Blom, H.A.P和Bar-Shalom, Y.，“用于具有Markovian切换系数的系统的交互多模型算法”，*IEEE自动控制杂志*，Vol. AC-8，No. 8，1998年8月，pp. 780-783。